# Aula 1 - LLM APIs

## Template da V1 do projeto

Neste notebook, você vai montar a primeira versão do assistente de estudos da monitoria.

As células com `TODO` são partes que você precisa completar. Não coloque sua API key diretamente no notebook, utilize o `.env`.

## Objetivos

Ao final, você deve conseguir:

- configurar um cliente para a API da Groq;
- montar mensagens com `system` e `user`;
- enviar uma requisição e ler a resposta;
- observar o uso de tokens;
- explicar por que chamadas separadas não têm memória automática;
- testar o efeito da temperatura;
- salvar a V1 em um commit.

## 1. Configuração do ambiente

Antes de executar o notebook:

1. crie um ambiente virtual;
2. instale as dependências de `requirements.txt`;
3. copie `env.example` para `.env`;
4. preencha `GROQ_API_KEY` no `.env`.

In [10]:
messages = [
    {"role": "system", "content": "Tu és um assistente para dúvidas de monitoria."},
    {"role": "user", "content": "Como posso tirar dúvidas na monitoria?"}
]

In [9]:
# Passo a passo para criar o .venv: 
# Abra o Terminal na raiz do projeto e digite:
# python -m venv venv
# Depois digite: 
# cd "Aula 01"
# pip install -r requirements.txt

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError(
        "GROQ_API_KEY não encontrada. Copie env.example para .env e informe sua chave."
    )

client = OpenAI(
    api_key=api_key,
    base_url=os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),
)

MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")
print(f"Modelo configurado: {MODEL}")

Modelo configurado: openai/gpt-oss-120b


## 2. Monte as mensagens

A aplicação controla o comportamento do assistente usando mensagens.

Use uma mensagem `system` para definir o papel do assistente e uma mensagem `user` para fazer uma pergunta sobre a monitoria.

In [6]:
messages = [
    {
        "role" : "system", 
        "content": "Voce é um assistente para dúvidas de monitoria."
    },
    {
        "role": "user",
        "content": "Como posso tirar dúvidas na monitoria?"
    }
]

## 3. Faça a primeira chamada

Envie `messages` para o modelo configurado em `MODEL` e salve a resposta em uma variável chamada `response`.

In [12]:
response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=0,
)

## 4. Leia o texto da resposta

A resposta contém uma lista de escolhas. Nesta atividade, vamos usar o texto da primeira escolha.

In [13]:

print(response.choices[0].message.content)

### Como tirar dúvidas na monitoria de forma eficaz

A monitoria é um espaço pensado para ajudar você a entender melhor o conteúdo, esclarecer pontos que ficaram confusos e avançar nos estudos. Para aproveitar ao máximo esse recurso, siga estas dicas:

---

## 1. Escolha o canal adequado

| Canal | Quando usar | Como acessar |
|-------|-------------|--------------|
| **Presencial (horário de plantão)** | Quando precisar de explicação rápida, demonstração ao vivo ou revisão de exercícios. | Verifique o calendário da disciplina e compareça ao local indicado (sala de monitoria, laboratório, etc.). |
| **Plataforma de mensagens (Slack, Discord, Teams)** | Dúvidas pontuais que podem ser respondidas em texto ou com arquivos anexados. | Entre no grupo da disciplina e procure o canal “#monitoria” ou “#dúvidas”. |
| **E‑mail ou formulário online** | Questões mais detalhadas, que exigem tempo de resposta ou anexos (código, PDFs). | Use o e‑mail oficial do monitor (geralmente disponibilizado no s

## 5. Observe o uso de tokens

Além do texto, a API informa quantos tokens entraram na chamada, quantos saíram e qual foi o total.

In [14]:
print("Tokens do prompt (entrada):", response.usage.prompt_tokens)
print("Tokens da resposta (saída):", response.usage.completion_tokens)
print("Total de tokens usados:", response.usage.total_tokens)

Tokens do prompt (entrada): 93
Tokens da resposta (saída): 1699
Total de tokens usados: 1792


## 6. Teste a ausência de memória

Faça uma chamada dizendo que o nome do seu usuário. Depois faça uma segunda chamada perguntando o nome, mas sem reenviar a primeira mensagem.

Salve a segunda resposta em `sem_historico`. O que aconteceu?

In [15]:

client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Olá! O meu nome é João."}
    ]
)


sem_historico = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Qual é o meu nome?"}
    ]
)


print(sem_historico.choices[0].message.content)

Eu não sei o seu nome. Poderia me dizer, por favor?


In [ ]:
# print(sem_historico.choices[0].message.content)

Agora repita a pergunta, mas envie explicitamente o histórico em `messages`. Inclua uma mensagem `assistant` representando a resposta anterior.

A aplicação cria a sensação de memória ao reenviar o contexto.

In [17]:

mensagens_com_historico = [
    {"role": "user", "content": "Olá! O meu nome é João."},
    {"role": "assistant", "content": "Olá João! Em que posso ajudar?"},
    {"role": "user", "content": "Qual é o meu nome?"}
]


com_historico = client.chat.completions.create(
    model=MODEL,
    messages=mensagens_com_historico
)

In [18]:
# Imprimir a resposta obtida com o histórico incluído
print(com_historico.choices[0].message.content)

O seu nome é João.


## 7. Experimente a temperatura

Use o mesmo prompt duas vezes: uma com `temperature=0` e outra com `temperature=1`.

Compare a variedade das respostas. Temperatura baixa tende a ser mais previsível; temperatura alta tende a permitir mais variação.

In [22]:

prompt_teste = [
    {"role": "user", "content": "Dá-me 3 ideias criativas de nomes para uma aplicação de inteligência artificial."}
]


resposta_temp_0 = client.chat.completions.create(
    model=MODEL,
    messages=prompt_teste,
    temperature=0
)


resposta_temp_1 = client.chat.completions.create(
    model=MODEL,
    messages=prompt_teste,
    temperature=1
)


print("temperatura 0 (Previsível)")
print(resposta_temp_0.choices[0].message.content)

print("\n" + "="*35 + "\n")

print("temperatura 1 (Variada/Criativa)")
print(resposta_temp_1.choices[0].message.content)

temperatura 0 (Previsível)
Claro! Aqui vão três sugestões criativas de nomes para a sua aplicação de inteligência artificial:

1. **NeuroPulse** – evoca a ideia de um “pulso” de inteligência que vibra em tempo real, combinando neurociência e tecnologia avançada.  
2. **CogniSphere** – sugere um universo de conhecimento interconectado, onde a IA pode explorar e responder a múltiplas áreas de forma holística.  
3. **AetherMind** – mistura o conceito clássico de “éter” (um meio invisível que permeia tudo) com “mind” (mente), transmitindo a sensação de uma inteligência onipresente e quase mágica.  

Espero que alguma delas inspire o seu projeto! 🚀


temperatura 1 (Variada/Criativa)
Claro! Aqui vão três sugestões criativas de nomes para a sua aplicação de inteligência artificial:

1. **NeuraFlux** – evoca a ideia de fluxos neurais e aprendizado contínuo, perfeito para uma IA que se adapta e evolui em tempo real.

2. **CogniSphere** – combina “cognição” com “esfera”, sugerindo uma plataforma

## 8. Checkpoint da V1

Antes de fazer o commit, confirme:

- [ x] a aplicação chamou a API da Groq;
- [x ] a chave ficou no `.env`;
- [x ] a requisição usou `system` e `user`;
- [x ] a resposta apareceu;
- [ x] o uso de tokens foi observado;
- [ x] você comparou chamadas sem histórico e com histórico;
- [ x] você testou duas temperaturas.
